Module 01: Exploratory Data Analysis for Demand & Inventory with Polars

This notebook performs exploratory data analysis (EDA) for Module 01 of the **"Intelligent System for Supply Chain Management"** project.  

The primary goal is to optimize inventory and purchasing management, with a target of **reducing overstocking by 20%** within six months.

---

## Import Libraries

In [1]:
import polars as pl
import polars.selectors as cs
import json
import plotly.express as px
import plotly.io as pio
from pathlib import Path
from polars_info import print_df_info
from datetime import date

# Install dependencies as needed:
# pip install kagglehub[polars-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter

import warnings
warnings.filterwarnings('ignore')

# Set up display options and plotting template
pio.templates.default = "plotly_white"
px.defaults.width = 800
px.defaults.height = 600

## Load Dataset

In [2]:
# Paths
path_docs = Path("..") / "docs"
path_processed = Path("..") / "data" / "processed"

In [3]:
# Set the path to the file you'd like to load
file_path = "1M_grocery_data_pl.parquet"

# Load the latest version
lf = kagglehub.dataset_load(
  KaggleDatasetAdapter.POLARS,
  "robertobalbinotti/synthetic-grocery-data",
  file_path,
  # Provide any additional arguments like
  # sql_query, polars_frame_type, or 
  # polars_kwargs.
  # See the documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpolars
)

In [7]:
# Load column descriptions from JSON file into a dictionary for reference or documentation
with open(path_docs / 'column_descriptions_polars.json') as f:
    column_descriptions = json.load(f)

# Data Cleaning and Preprocessing

In [8]:
print_df_info(lf.collect())

<class 'polars.dataframe.frame.DataFrame'>
Shape: (1,000,658, 31)
Estimated size: 182.25 MiB
Columns:
  #  Column                        Dtype         Non-Null    Null   Null%
  0  order_purchase_date           Date          1,000,658       0   0.00%
  1  received_date                 Date          1,000,658       0   0.00%
  2  product_id                    String        1,000,658       0   0.00%
  3  product                       String        1,000,658       0   0.00%
  4  category                      String        1,000,658       0   0.00%
  5  sub_category                  String        1,000,658       0   0.00%
  6  sales_demand                  String        1,000,658       0   0.00%
  7  sales_volume                  UInt16        1,000,658       0   0.00%
  8  seasonality                   List(String)  1,000,658       0   0.00%
  9  storage_recommendation        String        1,000,658       0   0.00%
 10  unit_of_measurement           String        1,000,658       0   0.00%

DFInfoSummary(rows=1000658, cols=31, estimated_size_bytes=191102740, dtypes={'order_purchase_date': Date, 'received_date': Date, 'product_id': String, 'product': String, 'category': String, 'sub_category': String, 'sales_demand': String, 'sales_volume': UInt16, 'seasonality': List(String), 'storage_recommendation': String, 'unit_of_measurement': String, 'shelf_life_days': UInt16, 'maximum_days_on_sale': UInt16, 'supplier_id': String, 'supplier': String, 'supplier_rating': UInt8, 'distance_km': UInt16, 'moq': UInt16, 'delivery_days': Float16, 'transit_time': Float16, 'in_season': Boolean, 'is_holiday': Boolean, 'day_classification': String, 'is_weekend': Boolean, 'min_stock': UInt16, 'max_stock': UInt16, 'stock_quantity': UInt16, 'temperature_classification': String, 'precipitation_classification': String, 'wind_classification': String, 'weather_severity': String})

In [9]:
df = lf.with_columns(
    pl.col(pl.Utf8).cast(pl.Categorical()),
)

df.show(2)

order_purchase_date,received_date,product_id,product,category,sub_category,sales_demand,sales_volume,seasonality,storage_recommendation,unit_of_measurement,shelf_life_days,maximum_days_on_sale,supplier_id,supplier,supplier_rating,distance_km,moq,delivery_days,transit_time,in_season,is_holiday,day_classification,is_weekend,min_stock,max_stock,stock_quantity,temperature_classification,precipitation_classification,wind_classification,weather_severity
date,date,cat,cat,cat,cat,cat,u16,list[str],cat,cat,u16,u16,cat,cat,u8,u16,u16,f16,f16,bool,bool,cat,bool,u16,u16,u16,cat,cat,cat,cat
2022-12-07,2022-12-09,"""1169187|P""","""Tomato""","""Fresh Foods""","""Vegetables""","""High""",164,"[""June"", ""July"", … ""September""]","""Room Temperature""","""lb""",7,3,"""1194877|S""","""ValleyFresh Farms""",4,85,100,0.605469,2.4140625,false,false,"""Weekday""",false,269,369,292,"""Warm""","""No precipitation""","""Gentle to Fresh Breeze""","""Moderate"""
2022-12-06,2022-12-09,"""1741974|P""","""Mozzarella Cheese""","""Dairy & Alternatives""","""Dairy""","""High""",108,[],"""Refrigerated""","""lb""",14,5,"""1422853|S""","""Artisan Cheesemakers""",5,95,40,0.931641,2.769531,false,false,"""Weekday""",false,237,277,275,"""Warm""","""No precipitation""","""Gentle to Fresh Breeze""","""Moderate"""


In [10]:
received_date = df.select("received_date").collect().to_series().sort().unique()

expected_date_range = pl.date_range(
    start= received_date.min(),
    end= received_date.max(),
    interval="1d",
    eager=True
)

is_complete = received_date.len() == expected_date_range.len()
missing_dates = expected_date_range.filter(~expected_date_range.is_in(received_date))

print("Complete Received Date Range?\n", is_complete)

Complete Received Date Range?
 True


# Feature Engineering

In [114]:
df = df.with_columns(
    (pl.col("received_date") - pl.col("order_purchase_date")).alias("delivery_lag")
).with_columns(
    pl.when(pl.col("delivery_lag") > pl.duration(days=pl.col("shelf_life_days")))
    .then(pl.lit("Expired"))
    .when(pl.col("delivery_lag") > pl.duration(days=pl.col("maximum_days_on_sale")))
    .then(pl.lit("Nearing"))
    .otherwise(pl.lit("Safe"))
    .cast(pl.Categorical())
    .alias("expiration_status")
).with_columns(
    (pl.col("order_purchase_date").dt.year())
    .cast(pl.UInt16).alias("year"),
    (pl.col("stock_quantity") - pl.col("sales_volume"))
    .cast(pl.Float32)
    .alias("closing_stock")
).with_columns(
    pl.col("sales_volume").sum().over(["product", "year"])
    .cast(pl.Float32)
    .alias("total_sales"),
    pl.col("closing_stock").mean().over(["product", "year"])
    .cast(pl.Float32)
    .alias("average_stock"), 
).with_columns(
    pl.when(pl.col("average_stock") > 0)
    .then(pl.col("total_sales") / pl.col("average_stock"))
    .otherwise(0.0)
    .cast(pl.Float32)
    .alias("inventory_turnover_rate")
).with_columns(
    pl.col("inventory_turnover_rate").mean().over(["product"])
    .cast(pl.Float32)
    .alias("average_turnover_rate")
).with_columns(
    (pl.col("received_date").max() - pl.col("received_date").min())
    .dt.total_days()
    .alias("period_days")
).with_columns(
    (pl.col("period_days") / pl.col("inventory_turnover_rate"))
    .floor()
    .cast(pl.UInt16)
    .alias("doi_inventory_turnover")
).drop(pl.col("period_days"))

df.show(2)

order_purchase_date,received_date,product_id,product,category,sub_category,sales_demand,sales_volume,seasonality,storage_recommendation,unit_of_measurement,shelf_life_days,maximum_days_on_sale,supplier_id,supplier,supplier_rating,distance_km,moq,delivery_days,transit_time,in_season,is_holiday,day_classification,is_weekend,min_stock,max_stock,stock_quantity,temperature_classification,precipitation_classification,wind_classification,weather_severity,delivery_lag,expiration_status,year,closing_stock,total_sales,average_stock,inventory_turnover_rate,average_turnover_rate,doi_inventory_turnover
date,date,cat,cat,cat,cat,cat,u16,list[str],cat,cat,u16,u16,cat,cat,u8,u16,u16,f16,f16,bool,bool,cat,bool,u16,u16,u16,cat,cat,cat,cat,duration[μs],cat,u16,f32,f32,f32,f32,f32,u16
2022-12-07,2022-12-09,"""1169187|P""","""Tomato""","""Fresh Foods""","""Vegetables""","""High""",164,"[""June"", ""July"", … ""September""]","""Room Temperature""","""lb""",7,3,"""1194877|S""","""ValleyFresh Farms""",4,85,100,0.605469,2.4140625,false,false,"""Weekday""",false,269,369,292,"""Warm""","""No precipitation""","""Gentle to Fresh Breeze""","""Moderate""",2d,"""Safe""",2022,128.0,38744.0,5642.839355,6.866047,67.31739,148
2022-12-06,2022-12-09,"""1741974|P""","""Mozzarella Cheese""","""Dairy & Alternatives""","""Dairy""","""High""",108,[],"""Refrigerated""","""lb""",14,5,"""1422853|S""","""Artisan Cheesemakers""",5,95,40,0.931641,2.769531,false,false,"""Weekday""",false,237,277,275,"""Warm""","""No precipitation""","""Gentle to Fresh Breeze""","""Moderate""",3d,"""Safe""",2022,167.0,12784.0,6806.277344,1.878266,23.328737,541


In [11]:
## Add a description for the 'inventory_turnover_rate' column
column_descriptions.update({
    "average_stock": "Mean inventory held during the period.",
    "average_turnover_rate": "Mean inventory turnover across periods/categories.",
    "closing_stock": "Remaining inventory at the end of the period.",
    "doi_inventory_turnover": "Days of inventory on hand (stock coverage).",
    "inventory_turnover_rate": "Times inventory is sold and replaced (COGS / Avg Stock).",
    "total_sales": "Total revenue or units sold during the period.",
    "year": "Calendar or fiscal year of the data."
})

# Exploratory Data Analysis (EDA)

### Distribution of Numerical Variables

In [ ]:
df_eda = df.collect()

num_cols = df_eda.select(cs.numeric()).columns

for col in num_cols:
    # Format column name
    ren_col = col.replace("_", " ").capitalize()
    # Create a histogram using Plotly Express for the current column
    fig = px.histogram(df_eda, x=col, title=f'Distribution of {ren_col}', labels={col: f"{col.replace("_", " ")}"}, nbins=30)
    
    # Adjust the gap between bars for better readability
    fig.update_layout(bargap=0.1)
    
    # Display the histogram
    fig.show()

### Distribution of Categorical Variables

In [63]:
cat_cols = (
    df_eda.select(
        cs.categorical() & ~cs.by_name("product_id", "supplier_id")
    ).columns
)

for col in cat_cols:
    # Format column name
    ren_col = col.replace("_", " ").capitalize()
    
    counts_df = df_eda[col].value_counts(sort=True).head(20)

    fig = px.bar(
        counts_df,
        x= col,
        y="count",
        title=f'Top 20 - Distribution of {ren_col}',
        labels={col: f"{col.replace("_", " ")}"}
    )
    
    fig.show()

### Relation Between Stock and Sales

In [ ]:
# Scatter plot: Stock vs Sales

# Create a scatter plot using the DataFrame 'df'
fig = px.scatter(
    df,  # Data source
    x='stock_quantity',  # X-axis represents stock quantity
    y='sales_volume',    # Y-axis represents sales volume
    color='sales_demand',  # Point color reflects sales demand
    title='Relation between Stock and Sales Volume',  # Chart title
    labels={  # Custom axis labels
        'stock_quantity': 'Stock',
        'sales_volume': 'Sales',
        'sales_demand': 'Demand'
    }
)

# Display the interactive chart
fig.show()

order_purchase_date,received_date,product_id,product,category,sub_category,sales_demand,sales_volume,seasonality,storage_recommendation,unit_of_measurement,shelf_life_days,maximum_days_on_sale,supplier_id,supplier,supplier_rating,distance_km,moq,delivery_days,transit_time,in_season,is_holiday,day_classification,is_weekend,min_stock,max_stock,stock_quantity,temperature_classification,precipitation_classification,wind_classification,weather_severity
date,date,cat,cat,cat,cat,cat,u16,list[str],cat,cat,u16,u16,cat,cat,u8,u16,u16,f16,f16,bool,bool,cat,bool,u16,u16,u16,cat,cat,cat,cat
2022-12-07,2022-12-09,"""1169187|P""","""Tomato""","""Fresh Foods""","""Vegetables""","""High""",164,"[""June"", ""July"", … ""September""]","""Room Temperature""","""lb""",7,3,"""1194877|S""","""ValleyFresh Farms""",4,85,100,0.605469,2.4140625,false,false,"""Weekday""",false,269,369,292,"""Warm""","""No precipitation""","""Gentle to Fresh Breeze""","""Moderate"""
2022-12-06,2022-12-09,"""1741974|P""","""Mozzarella Cheese""","""Dairy & Alternatives""","""Dairy""","""High""",108,[],"""Refrigerated""","""lb""",14,5,"""1422853|S""","""Artisan Cheesemakers""",5,95,40,0.931641,2.769531,false,false,"""Weekday""",false,237,277,275,"""Warm""","""No precipitation""","""Gentle to Fresh Breeze""","""Moderate"""
2022-12-06,2022-12-09,"""1113278|P""","""Carrot""","""Fresh Foods""","""Vegetables""","""High""",163,"[""January"", ""February"", … ""December""]","""Refrigerated""","""lb""",21,7,"""1194877|S""","""ValleyFresh Farms""",4,85,100,1.4140625,2.173828,true,false,"""Weekday""",false,269,369,282,"""Warm""","""No precipitation""","""Gentle to Fresh Breeze""","""Moderate"""
2022-12-05,2022-12-09,"""1234660|P""","""Butter""","""Dairy & Alternatives""","""Dairy""","""High""",111,[],"""Refrigerated""","""unit""",30,10,"""1176804|S""","""Daily Dairy""",4,55,85,1.421875,1.375977,false,false,"""Weekday""",false,195,280,216,"""Warm""","""No precipitation""","""Gentle to Fresh Breeze""","""Moderate"""
2022-12-06,2022-12-09,"""1202127|P""","""Pomegranate""","""Fresh Foods""","""Fruits""","""High""",168,"[""September"", ""October"", … ""December""]","""Room Temperature""","""unit""",21,7,"""1211763|S""","""Tropical Fruits Ltd.""",4,350,80,1.963867,19.859375,true,false,"""Weekday""",false,439,519,500,"""Warm""","""No precipitation""","""Gentle to Fresh Breeze""","""Moderate"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2025-09-19,2025-09-22,"""1230940|P""","""Haddock""","""Fresh Foods""","""Seafood""","""Normal""",6,[],"""Refrigerated""","""lb""",2,1,"""1891168|S""","""OceanHarvest Seafood""",2,180,40,0.788086,3.341797,false,false,"""Weekday""",false,23,63,118,"""Mild to Temperate""","""Heavy Rain""","""Gentle to Fresh Breeze""","""Severe"""
2025-09-18,2025-09-22,"""1385860|P""","""Granola Bars""","""Pantry""","""Snacks""","""Normal""",78,[],"""Room Temperature""","""unit""",90,30,"""1783257|S""","""SnackTime Distributors""",5,80,110,1.323242,2.681641,false,false,"""Weekday""",false,203,313,0,"""Mild to Temperate""","""Heavy Rain""","""Gentle to Fresh Breeze""","""Severe"""
2025-09-20,2025-09-22,"""1543068|P""","""Rye Bread""","""Bakery""","""Bread""","""Normal""",109,[],"""Room Temperature""","""unit""",5,2,"""1322480|S""","""Bakery Fresh Co.""",2,45,75,1.256836,1.758789,false,false,"""Weekday""",false,193,268,252,"""Mild to Temperate""","""Heavy Rain""","""Gentle to Fresh Breeze""","""Severe"""


In [17]:
set(df_eda.dtypes)

{Boolean, Categorical, Date, Float16, List(String), UInt16, UInt8}